# LeetCode 337 — House Robber III bằng Python

Link đề: https://leetcode.com/problems/house-robber-iii/ · Độ khó: **Medium** · Chủ đề: Dynamic Programming, Tree, Depth-First Search, Binary Tree, Tree DP

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def rob(self, root: Optional[TreeNode]) -> int:
```

Ghi chú: đề, ví dụ và constraints lấy từ file đề có sẵn (bài này không phải Premium). LeetCode tự cung cấp `TreeNode`; trong notebook ta tự dựng `TreeNode` và hàm `build_tree` (mảng theo tầng, `None` cho nút rỗng) để test.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: các nhà tạo thành **cây nhị phân**; **không được cướp hai nhà nối trực tiếp** (cha–con) trong cùng một đêm; tìm tổng tiền lớn nhất — đây là bài **tập độc lập có trọng số lớn nhất trên cây**.
2. Phân biệt với **House Robber I** (dãy thẳng): ở đây "kề nhau" nghĩa là **cha–con**, và mỗi nút có **hai** con.
3. Đi từ **đệ quy "cướp / không cướp"** (có trùng lặp tính toán theo dạng Fibonacci) → **ghi nhớ (memo) theo nút** → **DP hậu thứ tự trả về cặp `(cướp, không cướp)`** `O(n)` → **bản lặp** → **duyệt theo tầng rồi gộp ngược từ lá lên gốc (không đệ quy)**.
4. Hiểu **vì sao cặp hai trạng thái** là đủ: mỗi nút chỉ cần biết "tốt nhất nếu **cướp** nó" và "tốt nhất nếu **bỏ qua** nó".
5. Biết kiểm chứng bằng **oracle vét cạn mọi tập con** (độc lập) và, với cây dạng xích, bằng bài **House Robber I** (1 chiều).
6. Tránh các lỗi kinh điển: chia theo tầng chẵn/lẻ, luôn cướp nút hiện tại, tính sai trạng thái "bỏ qua", ghi nhớ theo **giá trị** thay vì **nút**, và đệ quy quá sâu.

## 2. Đề bài & ví dụ

Tên trộm tìm được một khu mới. Chỉ có một lối vào gọi là `root`. Ngoài `root`, mỗi nhà có **đúng một nhà cha**. Sau khi quan sát, hắn nhận ra mọi nhà tạo thành một **cây nhị phân**. Hệ thống tự động báo cảnh sát nếu **hai nhà nối trực tiếp** (cha–con) bị đột nhập trong cùng một đêm.

Cho `root` của cây, trả về **số tiền lớn nhất** tên trộm có thể lấy mà **không báo động**.

**Ví dụ 1:** `root = [3,2,3,null,3,null,1]` → `7` (cướp `3 + 3 + 1`).

**Ví dụ 2:** `root = [3,4,5,1,3,null,1]` → `9` (cướp `4 + 5`).

**Ràng buộc:** số nút trong khoảng `[1, 10^4]`; `0 <= Node.val <= 10^4`.

In [1]:
from typing import Optional, List
from collections import deque
import random
import sys
import time


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    """Dựng cây từ mảng theo tầng (None = nút rỗng) như định dạng LeetCode."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def to_level_order(root):
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


examples = [
    ([3, 2, 3, None, 3, None, 1], 7),
    ([3, 4, 5, 1, 3, None, 1], 9),
]
for values, expected in examples:
    print(values, "| dựng lại:", to_level_order(build_tree(values)), "| kỳ vọng:", expected)

[3, 2, 3, None, 3, None, 1] | dựng lại: [3, 2, 3, None, 3, None, 1] | kỳ vọng: 7
[3, 4, 5, 1, 3, None, 1] | dựng lại: [3, 4, 5, 1, 3, None, 1] | kỳ vọng: 9


## 3. Kiến thức nền

### 3.1. House Robber I (dãy thẳng) — điểm xuất phát

Với dãy nhà `a[0..n-1]`, không cướp hai nhà liền kề: `dp[i] = max(dp[i-1], dp[i-2] + a[i])` (bỏ qua nhà `i`, hoặc cướp nhà `i` cộng với tốt nhất tới `i-2`). Cây là sự tổng quát hóa: "liền kề" = cha–con.
#### Code demo

In [2]:
def rob_line(a):
    take, skip = 0, 0                    # take: tốt nhất nếu cướp nhà cuối; skip: tốt nhất nếu bỏ nhà cuối
    for x in a:
        take, skip = skip + x, max(take, skip)
    return max(take, skip)


print(rob_line([1, 2, 3, 1]), rob_line([2, 7, 9, 3, 1]))     # 4 và 12

4 12


### 3.2. Hai trạng thái trên mỗi nút

Với mỗi nút `u` ta xét **hai** đại lượng cho **cây con gốc `u`**:

- `rob(u)`: tổng lớn nhất **khi cướp `u`**. Khi đó hai con **không được cướp**: `rob(u) = u.val + skip(left) + skip(right)`.
- `skip(u)`: tổng lớn nhất **khi không cướp `u`**. Khi đó mỗi con **tự do chọn** cướp hay bỏ: `skip(u) = max(rob(left), skip(left)) + max(rob(right), skip(right))`.

Đáp án là `max(rob(root), skip(root))`. Với cây rỗng: `rob = skip = 0`.
#### Code demo

In [3]:
# Minh họa trên cây [3, 4, 5]: gốc 3, con trái 4, con phải 5 (đều là lá)
leaf4 = (4, 0)      # (rob, skip) của lá 4
leaf5 = (5, 0)
rob_root = 3 + leaf4[1] + leaf5[1]
skip_root = max(leaf4) + max(leaf5)
print("rob(root) =", rob_root, "| skip(root) =", skip_root, "| đáp án =", max(rob_root, skip_root))

rob(root) = 3 | skip(root) = 9 | đáp án = 9


### 3.3. Hậu thứ tự (post-order)

Muốn tính `rob(u)` và `skip(u)` ta cần kết quả của hai con trước, tức xử lý **con trước, cha sau** — thứ tự **hậu thứ tự**. Có thể làm bằng đệ quy hoặc bằng ngăn xếp / duyệt theo tầng đảo ngược.
#### Code demo

In [4]:
def postorder_values(node):
    if node is None:
        return []
    return postorder_values(node.left) + postorder_values(node.right) + [node.val]


print(postorder_values(build_tree([3, 2, 3, None, 3, None, 1])))     # [3, 2, 1, 3, 3]

[3, 2, 1, 3, 3]


### 3.4. Giới hạn đệ quy của Python

Cây có tới `10^4` nút có thể sâu `10^4` (xích). Mặc định Python cho đệ quy sâu ~`1000`. Ta nâng bằng `sys.setrecursionlimit(...)` hoặc viết bản **lặp**.
#### Code demo

In [5]:
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit())
sys.setrecursionlimit(100_000)
print("sau khi nâng:", sys.getrecursionlimit())

giới hạn đệ quy mặc định: 1000
sau khi nâng: 100000


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ 1: `[3, 2, 3, None, 3, None, 1]`

```
      3
     / \
    2   3
     \   \
      3   1
```

Tính từ dưới lên, ghi `(rob, skip)`:

| nút | con trái | con phải | `rob = val + skip_L + skip_R` | `skip = max(L) + max(R)` |
|---|---|---|---|---|
| `3` (con của 2) | rỗng | rỗng | 3 | 0 |
| `1` (con của 3) | rỗng | rỗng | 1 | 0 |
| `2` | rỗng | `(3, 0)` | `2 + 0 + 0 = 2` | `0 + max(3, 0) = 3` |
| `3` (phải) | rỗng | `(1, 0)` | `3 + 0 + 0 = 3` | `0 + max(1, 0) = 1` |
| `3` (gốc) | `(2, 3)` | `(3, 1)` | `3 + 3 + 1 = 7` | `max(2, 3) + max(3, 1) = 6` |

Đáp án `max(7, 6) = 7` — khớp: cướp gốc `3` cùng `3` (cháu trái) và `1` (cháu phải).

## 5. Cách 1 — Brute Force / baseline: đệ quy "cướp / không cướp" không ghi nhớ

**Ý tưởng.** `f(u) = max( u.val + f(các cháu của u),  f(left) + f(right) )`: hoặc cướp `u` (khi đó bỏ hai con, tiếp tục từ **cháu**), hoặc không cướp `u` (tiếp tục từ hai con). Không có bảng nhớ nên nhiều cây con bị tính lại.
#### Code demo

In [6]:
def rob_naive(root):
    def f(u):
        if u is None:
            return 0
        take = u.val
        if u.left:
            take += f(u.left.left) + f(u.left.right)
        if u.right:
            take += f(u.right.left) + f(u.right.right)
        skip = f(u.left) + f(u.right)
        return max(take, skip)

    return f(root)


for values, expected in examples:
    print(values, "->", rob_naive(build_tree(values)), "| kỳ vọng:", expected)

[3, 2, 3, None, 3, None, 1] -> 7 | kỳ vọng: 7
[3, 4, 5, 1, 3, None, 1] -> 9 | kỳ vọng: 9


**Độ phức tạp (Cách 1).** **Time** tăng **theo hàm mũ** (kiểu Fibonacci: một nút được thăm cả khi là "con" của cha lẫn khi là "cháu" của ông), **Space `O(h)`** (ngăn xếp). Với xích `n` nút số lần gọi cỡ `φ^n` (xem mục 13).

**Nút thắt.** `f(u)` bị tính lại nhiều lần; mỗi nút chỉ cần tính **một lần**.

## 6. Ý tưởng thuật toán / bước đột phá

**Tính cả hai trạng thái trong một lượt hậu thứ tự.** Hàm `dfs(u)` trả về cặp `(rob, skip)` của cây con `u`:

```
rob  = u.val + skip_L + skip_R
skip = max(rob_L, skip_L) + max(rob_R, skip_R)
```

Mỗi nút tính đúng **một lần** với hằng số phép ⇒ **`O(n)`**. Đáp án: `max(dfs(root))`.

## 7. Các cách tiếp theo

### Cách 2 — Đệ quy có ghi nhớ (memo theo nút)

**Giải thích.** Giữ nguyên công thức `f(u)` của Cách 1 nhưng lưu kết quả trong từ điển **khóa theo nút** (`id(u)`), để mỗi nút chỉ tính một lần. Khóa phải là **danh tính nút**, không phải giá trị `val` (hai nút khác nhau có thể cùng giá trị — xem Lỗi 4).
#### Code demo

In [7]:
def rob_memo(root):
    memo = {}

    def f(u):
        if u is None:
            return 0
        key = id(u)
        if key in memo:
            return memo[key]
        take = u.val
        if u.left:
            take += f(u.left.left) + f(u.left.right)
        if u.right:
            take += f(u.right.left) + f(u.right.right)
        memo[key] = max(take, f(u.left) + f(u.right))
        return memo[key]

    return f(root)


for values, expected in examples:
    print(values, "->", rob_memo(build_tree(values)), "| kỳ vọng:", expected)

[3, 2, 3, None, 3, None, 1] -> 7 | kỳ vọng: 7
[3, 4, 5, 1, 3, None, 1] -> 9 | kỳ vọng: 9


**Độ phức tạp Cách 2:** **Time `O(n)`** (mỗi nút tính một lần), **Space `O(n)`** (bảng memo + ngăn xếp).

### Cách 3 — DP hậu thứ tự trả về cặp `(cướp, không cướp)` (cách nộp chuẩn)

**Giải thích.** Như mục 6.
#### Code demo

In [8]:
def rob_pair(root):
    def dfs(u):
        if u is None:
            return 0, 0                                   # (rob, skip) của cây rỗng
        l_rob, l_skip = dfs(u.left)
        r_rob, r_skip = dfs(u.right)
        rob = u.val + l_skip + r_skip                     # cướp u: hai con phải bỏ
        skip = max(l_rob, l_skip) + max(r_rob, r_skip)    # bỏ u: mỗi con tự chọn tốt nhất
        return rob, skip

    return max(dfs(root))


for values, expected in examples:
    print(values, "->", rob_pair(build_tree(values)), "| kỳ vọng:", expected)

[3, 2, 3, None, 3, None, 1] -> 7 | kỳ vọng: 7
[3, 4, 5, 1, 3, None, 1] -> 9 | kỳ vọng: 9


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(h)`** ngăn xếp đệ quy (`h` là chiều cao, tối đa `n`).

### Cách 4 — Hậu thứ tự lặp (ngăn xếp tường minh)

**Giải thích.** Dùng ngăn xếp `(nút, đã_xử_lý_con?)`. Lần đầu gặp nút: đẩy lại nó với cờ `True` rồi đẩy hai con; lần sau (con đã xong) tính cặp `(rob, skip)` từ bảng kết quả theo `id`. Không đệ quy ⇒ không bị giới hạn độ sâu.
#### Code demo

In [9]:
def rob_iterative(root):
    if root is None:
        return 0
    res = {}                                              # id(nút) -> (rob, skip)
    stack = [(root, False)]
    while stack:
        u, seen = stack.pop()
        if u is None:
            continue
        if not seen:
            stack.append((u, True))
            stack.append((u.right, False))
            stack.append((u.left, False))
            continue
        l_rob, l_skip = res[id(u.left)] if u.left else (0, 0)
        r_rob, r_skip = res[id(u.right)] if u.right else (0, 0)
        res[id(u)] = (u.val + l_skip + r_skip, max(l_rob, l_skip) + max(r_rob, r_skip))
    return max(res[id(root)])


for values, expected in examples:
    print(values, "->", rob_iterative(build_tree(values)), "| kỳ vọng:", expected)

[3, 2, 3, None, 3, None, 1] -> 7 | kỳ vọng: 7
[3, 4, 5, 1, 3, None, 1] -> 9 | kỳ vọng: 9


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(n)`**, **không đệ quy**.

### Cách 5 — Duyệt theo tầng (BFS) rồi gộp ngược từ lá lên gốc

**Giải thích.** Duyệt BFS ghi thứ tự các nút và chỉ số cha. Đặt `rob[u] = u.val`, `skip[u] = 0`. Duyệt **ngược** thứ tự BFS (con luôn đứng sau cha trong BFS nên khi đảo ngược con đứng trước cha): khi tới `u` thì mọi con của `u` đã cộng xong vào `u`; ta đẩy kết quả của `u` lên cha: `rob[p] += skip[u]` và `skip[p] += max(rob[u], skip[u])`. Hoàn toàn không đệ quy, không cần ngăn xếp phức tạp.
#### Code demo

In [10]:
def rob_bfs_merge(root):
    if root is None:
        return 0
    order, parent = [], []
    queue = deque([(root, -1)])
    while queue:
        node, p = queue.popleft()
        idx = len(order)
        order.append(node)
        parent.append(p)
        if node.left:
            queue.append((node.left, idx))
        if node.right:
            queue.append((node.right, idx))
    n = len(order)
    rob = [node.val for node in order]
    skip = [0] * n
    for i in range(n - 1, 0, -1):                          # từ lá lên; bỏ qua gốc (i = 0)
        p = parent[i]
        rob[p] += skip[i]
        skip[p] += max(rob[i], skip[i])
    return max(rob[0], skip[0])


for values, expected in examples:
    print(values, "->", rob_bfs_merge(build_tree(values)), "| kỳ vọng:", expected)

[3, 2, 3, None, 3, None, 1] -> 7 | kỳ vọng: 7
[3, 4, 5, 1, 3, None, 1] -> 9 | kỳ vọng: 9


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(n)`**, không đệ quy.

> Cách 3 là bản nộp thông dụng (ngắn, dễ đọc). Cách 4/5 an toàn khi cây rất sâu. Cách 2 minh họa "đệ quy + memo" trước khi gộp hai trạng thái.

## 8. TRACE từng bước

Ta trace Cách 3 trên ví dụ 2 `[3, 4, 5, 1, 3, None, 1]`: in nút khi hoàn tất theo hậu thứ tự, kèm `(rob, skip)` của hai con.
#### Code demo

In [11]:
def trace_pair(root):
    def dfs(u, depth=0):
        pad = "  " * depth
        if u is None:
            return 0, 0
        l_rob, l_skip = dfs(u.left, depth + 1)
        r_rob, r_skip = dfs(u.right, depth + 1)
        rob = u.val + l_skip + r_skip
        skip = max(l_rob, l_skip) + max(r_rob, r_skip)
        print(f"{pad}nút {u.val}: trái(rob={l_rob}, skip={l_skip}) phải(rob={r_rob}, skip={r_skip}) "
              f"=> rob = {u.val}+{l_skip}+{r_skip} = {rob} | skip = max({l_rob},{l_skip})+max({r_rob},{r_skip}) = {skip}")
        return rob, skip

    r, s = dfs(root)
    print(f"đáp án = max(rob={r}, skip={s}) = {max(r, s)}")
    return max(r, s)


trace_pair(build_tree([3, 4, 5, 1, 3, None, 1]))

    nút 1: trái(rob=0, skip=0) phải(rob=0, skip=0) => rob = 1+0+0 = 1 | skip = max(0,0)+max(0,0) = 0
    nút 3: trái(rob=0, skip=0) phải(rob=0, skip=0) => rob = 3+0+0 = 3 | skip = max(0,0)+max(0,0) = 0
  nút 4: trái(rob=1, skip=0) phải(rob=3, skip=0) => rob = 4+0+0 = 4 | skip = max(1,0)+max(3,0) = 4
    nút 1: trái(rob=0, skip=0) phải(rob=0, skip=0) => rob = 1+0+0 = 1 | skip = max(0,0)+max(0,0) = 0
  nút 5: trái(rob=0, skip=0) phải(rob=1, skip=0) => rob = 5+0+0 = 5 | skip = max(0,0)+max(1,0) = 1
nút 3: trái(rob=4, skip=4) phải(rob=5, skip=1) => rob = 3+4+1 = 8 | skip = max(4,4)+max(5,1) = 9
đáp án = max(rob=8, skip=9) = 9


9

Bảng tóm tắt (ví dụ 2):

| nút | `(rob, skip)` |
|---|---|
| `1` (con của 4) | `(1, 0)` |
| `3` (con của 4) | `(3, 0)` |
| `1` (con của 5) | `(1, 0)` |
| `4` | `(4 + 0 + 0, max(1,0) + max(3,0)) = (4, 4)` |
| `5` | `(5 + 0, 0 + max(1,0)) = (5, 1)` |
| `3` (gốc) | `rob = 3 + 4 + 1 = 8`, `skip = max(4,4) + max(5,1) = 9` ⇒ **9** |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3. Đúng `class Solution`, đúng `rob(self, root: Optional[TreeNode]) -> int`. (Nếu lo độ sâu `10^4`, thêm `sys.setrecursionlimit(...)` hoặc dùng bản lặp của Cách 4.)
#### Code demo

In [12]:
class Solution:
    def rob(self, root: Optional[TreeNode]) -> int:
        def dfs(node):
            # trả về (số tiền tối đa nếu cướp node, số tiền tối đa nếu bỏ node)
            if node is None:
                return 0, 0
            l_rob, l_skip = dfs(node.left)
            r_rob, r_skip = dfs(node.right)
            rob = node.val + l_skip + r_skip                       # cướp node => hai con phải bỏ
            skip = max(l_rob, l_skip) + max(r_rob, r_skip)         # bỏ node => mỗi con tự chọn
            return rob, skip

        return max(dfs(root))


sol = Solution()
for values, expected in examples:
    assert sol.rob(build_tree(values)) == expected, (values, expected)
print("OK: 2 ví dụ của đề")

OK: 2 ví dụ của đề


## 10. Vì sao thuật toán đúng

Gọi `R(u)`, `S(u)` là số tiền lớn nhất trong cây con gốc `u` **khi cướp `u`** và **khi không cướp `u`**. Ta chứng minh bằng quy nạp theo chiều cao rằng `dfs(u) = (R(u), S(u))`.

- **Cây rỗng:** không có nhà nào ⇒ `(0, 0)`.
- **`R(u)`:** nếu cướp `u` thì không được cướp hai con (nối trực tiếp với `u`). Các cây con của hai con **độc lập** nhau (không nối trực tiếp giữa cây con trái và phải) nên tối ưu hóa riêng từng bên: trái bỏ `left` được tối ưu `S(left)`; phải bỏ `right` được tối ưu `S(right)`. Vậy `R(u) = u.val + S(left) + S(right)`.
- **`S(u)`:** nếu không cướp `u` thì hai con **không còn bị ràng buộc bởi `u`**, mỗi con chọn tốt nhất trong hai trạng thái: `max(R(left), S(left)) + max(R(right), S(right))`; hai nhánh độc lập.
- **Đáp án:** tối ưu toàn cây = `max(R(root), S(root))` vì gốc hoặc được cướp hoặc không.
- **Cách 1 / Cách 2:** `f(u) = max(R(u), S(u))`. Khi cướp `u` thì phần còn lại tối ưu là `f` của **cháu** (vì con bị bỏ, cháu tự do): `R(u) = u.val + f(cháu)`. Khi bỏ `u`: `S(u) = f(left) + f(right)`. Công thức khớp với cặp `(R, S)` nên đúng; ghi nhớ chỉ tránh tính lặp.
- **Cách 4 / Cách 5:** cùng công thức, chỉ khác thứ tự thăm (đảm bảo con xử lý trước cha).

## 11. Phân tích Time / Space Complexity

Gọi `n` là số nút (≤ `10^4`), `h` là chiều cao.

| Cách | Time | Space |
|---|---|---|
| 1. Đệ quy không memo | mũ (kiểu Fibonacci) | `O(h)` |
| 2. Đệ quy + memo theo nút | `O(n)` | `O(n)` |
| 3. DP cặp `(rob, skip)` | `O(n)` | `O(h)` |
| 4. Hậu thứ tự lặp | `O(n)` | `O(n)` |
| 5. BFS rồi gộp ngược | `O(n)` | `O(n)` |

Với `n = 10^4`: các cách tuyến tính chỉ khoảng `10^4` bước; Cách 1 với xích `30` nút đã cỡ hàng triệu lần gọi. `h` có thể bằng `n` (xích) nên đệ quy cần nâng giới hạn hoặc dùng bản lặp.

## 12. Bộ test

`oracle(root)` **độc lập**: **vét cạn mọi tập con** các nút (với `n ≤ 10`), loại các tập có cặp cha–con cùng được chọn, lấy tổng lớn nhất. Cho cây dạng xích dùng thêm **House Robber I** (`rob_line`) để kiểm tra cây lớn.
#### Code demo

In [13]:
def all_nodes_with_parent(root):
    """Trả (danh sách nút theo BFS, danh sách chỉ số cha)."""
    order, parent = [], []
    queue = deque([(root, -1)])
    while queue:
        node, p = queue.popleft()
        idx = len(order)
        order.append(node)
        parent.append(p)
        if node.left:
            queue.append((node.left, idx))
        if node.right:
            queue.append((node.right, idx))
    return order, parent


def oracle(root):
    """Vét cạn mọi tập con (n nhỏ): không được chọn cả cha lẫn con."""
    if root is None:
        return 0
    order, parent = all_nodes_with_parent(root)
    n = len(order)
    vals = [node.val for node in order]
    edges = [(i, parent[i]) for i in range(1, n)]
    best = 0
    for mask in range(1 << n):
        if any((mask >> i) & 1 and (mask >> p) & 1 for i, p in edges):
            continue
        total = sum(vals[i] for i in range(n) if (mask >> i) & 1)
        if total > best:
            best = total
    return best


impls = {
    "naive": rob_naive,
    "memo": rob_memo,
    "pair": rob_pair,
    "iterative": rob_iterative,
    "bfs_merge": rob_bfs_merge,
    "submit": sol.rob,
}

for values, expected in examples:
    assert oracle(build_tree(values)) == expected
    for key, fn in impls.items():
        assert fn(build_tree(values)) == expected, (key, values, expected, fn(build_tree(values)))
print("PASS ví dụ của đề (cả oracle)")

PASS ví dụ của đề (cả oracle)


### 12.1. Các trường hợp biên
#### Code demo

In [14]:
edge_cases = [
    ([5], 5, "một nút"),
    ([0], 0, "một nút giá trị 0"),
    ([1, 100], 100, "cha nhỏ, con lớn"),
    ([100, 1], 100, "cha lớn, con nhỏ"),
    ([2, 1, 3, None, 4], 7, "3 + 4 (hai nút không kề nhau)"),
    ([4, 1, None, 2, None, 3], 7, "xích trái: 4 + 3 (bỏ 1 và 2)"),
    ([1, 2, 3, 4, 5, 6, 7], 23, "cây đầy đủ 3 tầng: 1 + 4 + 5 + 6 + 7"),
    ([10, 1, 1, 1, 1, 1, 1], 10 + 4, "gốc lớn, các cháu mỗi cái 1"),
    ([0, 5, 5], 10, "gốc 0: cướp hai con"),
    ([5, 0, 0, 5, 5, 5, 5], 25, "bỏ hàng giữa toàn 0: gốc 5 + bốn cháu 5"),
    ([3, 3, 3, 3, 3, 3, 3], 15, "mọi giá trị bằng nhau (danh tính nút phân biệt)"),
    ([10 ** 4] * 7, 5 * 10 ** 4, "giá trị lớn nhất"),
]
for values, expected, label in edge_cases:
    t = build_tree(values)
    o = oracle(t)
    assert o == expected, (label, "oracle khác kỳ vọng tay", o, expected)
    for key, fn in impls.items():
        assert fn(build_tree(values)) == expected, (key, label, values, expected, fn(build_tree(values)))
    print("PASS", label, "->", expected)

PASS một nút -> 5
PASS một nút giá trị 0 -> 0
PASS cha nhỏ, con lớn -> 100
PASS cha lớn, con nhỏ -> 100
PASS 3 + 4 (hai nút không kề nhau) -> 7
PASS xích trái: 4 + 3 (bỏ 1 và 2) -> 7
PASS cây đầy đủ 3 tầng: 1 + 4 + 5 + 6 + 7 -> 23
PASS gốc lớn, các cháu mỗi cái 1 -> 14
PASS gốc 0: cướp hai con -> 10
PASS bỏ hàng giữa toàn 0: gốc 5 + bốn cháu 5 -> 25
PASS mọi giá trị bằng nhau (danh tính nút phân biệt) -> 15
PASS giá trị lớn nhất -> 50000


### 12.2. Tính chất và phản ví dụ

- Đáp án `≥` giá trị lớn nhất của một nút, và `≥` tổng các nút ở **một tầng bất kỳ** (các nút cùng tầng không nối trực tiếp với nhau).
- Đáp án `≥` tổng các **tầng chẵn** và `≥` tổng các **tầng lẻ** (tầng cách nhau không kề nhau).
- Đáp án `≤` tổng mọi giá trị.
- **Cây xích** trùng với **House Robber I** trên dãy giá trị theo chiều sâu.
- Nhân mọi giá trị với `c ≥ 0` thì đáp án nhân `c`.
- Đáp án **không** luôn bằng `max(tổng tầng chẵn, tổng tầng lẻ)` → **phản ví dụ** `[4, 1, None, 2, None, 3]`.
#### Code demo

In [15]:
def random_tree(rng, n, hi):
    """Cây ngẫu nhiên n nút với giá trị trong [0, hi] (gắn nút mới vào một ô trống ngẫu nhiên)."""
    if n == 0:
        return None
    nodes = [TreeNode(rng.randint(0, hi)) for _ in range(n)]
    slots = [(0, "left"), (0, "right")]
    for i in range(1, n):
        idx = rng.randrange(len(slots))
        slots[idx], slots[-1] = slots[-1], slots[idx]
        parent, side = slots.pop()
        setattr(nodes[parent], side, nodes[i])
        slots.extend([(i, "left"), (i, "right")])
    return nodes[0]


def level_sums(root):
    sums, level = [], [root] if root else []
    while level:
        sums.append(sum(node.val for node in level))
        level = [c for node in level for c in (node.left, node.right) if c]
    return sums


rng = random.Random(337)
for _ in range(400):
    n = rng.randint(1, 10)
    t = random_tree(rng, n, 9)
    ans = sol.rob(t)
    order, _ = all_nodes_with_parent(t)
    vals = [node.val for node in order]
    sums = level_sums(t)
    assert ans >= max(vals) and ans <= sum(vals)
    assert ans >= sum(sums[0::2]) and ans >= sum(sums[1::2])
    assert all(ans >= s for s in sums)
    for node in order:
        node.val *= 3
    assert sol.rob(t) == 3 * ans
print("PASS: giới hạn (max nút, tổng), >= tổng các tầng cách nhau, nhân hằng số")


def sai_chan_le(root):
    """SAI: chỉ so sánh tổng các tầng chẵn với tổng các tầng lẻ."""
    sums = level_sums(root)
    return max(sum(sums[0::2]), sum(sums[1::2]))


chain = build_tree([4, 1, None, 2, None, 3])
print("tầng chẵn/lẻ:", sai_chan_le(chain), "| đúng:", sol.rob(chain))
assert sai_chan_le(chain) != sol.rob(chain)

PASS: giới hạn (max nút, tổng), >= tổng các tầng cách nhau, nhân hằng số
tầng chẵn/lẻ: 6 | đúng: 7


### 12.3. Kiểm tra ngẫu nhiên (có seed) và dữ liệu lớn
#### Code demo

In [16]:
rng = random.Random(2337)
for trial in range(1500):
    n = rng.randint(1, 10)
    hi = rng.choice([1, 3, 10, 10 ** 4])
    t = random_tree(rng, n, hi)
    expected = oracle(t)
    for key, fn in impls.items():
        assert fn(t) == expected, (key, to_level_order(t), expected, fn(t))
print("PASS 1500 cây ngẫu nhiên với oracle vét cạn tập con")


def robber_line_chain(values, direction):
    root = TreeNode(values[0])
    cur = root
    for v in values[1:]:
        node = TreeNode(v)
        setattr(cur, direction, node)
        cur = node
    return root


def balanced(vals):
    if not vals:
        return None
    m = len(vals) // 2
    return TreeNode(vals[m], balanced(vals[:m]), balanced(vals[m + 1:]))


# So sánh cây lớn: các cách tuyến tính với nhau + House Robber I trên xích
N = 10_000
rng = random.Random(5)
chain_vals = [rng.randint(0, 10 ** 4) for _ in range(N)]
big_cases = [
    ("xích trái (so với House Robber I)", robber_line_chain(chain_vals, "left"), rob_line(chain_vals)),
    ("xích phải (so với House Robber I)", robber_line_chain(chain_vals, "right"), rob_line(chain_vals)),
    ("xích giá trị 1 (đáp án n/2)", robber_line_chain([1] * N, "left"), N // 2),
    ("cây cân bằng", balanced([rng.randint(0, 10 ** 4) for _ in range(N)]), None),
    ("cây ngẫu nhiên", random_tree(random.Random(9), N, 10 ** 4), None),
    ("mọi giá trị 0, cây ngẫu nhiên (đáp án 0)", random_tree(random.Random(10), N, 0), 0),
]
for label, t, expected in big_cases:
    row, answers = [], set()
    for key in ("memo", "pair", "iterative", "bfs_merge", "submit"):
        t0 = time.perf_counter()
        r = impls[key](t)
        dt = (time.perf_counter() - t0) * 1000
        answers.add(r)
        row.append(f"{key}={dt:6.1f}ms")
    assert len(answers) == 1, (label, answers)
    ans = answers.pop()
    if expected is not None:
        assert ans == expected, (label, ans, expected)
    print(f"{label:<36} ans={ans:>9} | " + " ".join(row))
print("PASS n = 10^4")

PASS 1500 cây ngẫu nhiên với oracle vét cạn tập con
xích trái (so với House Robber I)    ans= 29203487 | memo=   4.7ms pair=   3.1ms iterative=   5.0ms bfs_merge=   2.1ms submit=   2.7ms
xích phải (so với House Robber I)    ans= 29203487 | memo=   4.3ms pair=   2.5ms iterative=   4.0ms bfs_merge=   2.1ms submit=   2.4ms
xích giá trị 1 (đáp án n/2)          ans=     5000 | memo=   3.9ms pair=   2.3ms iterative=   4.2ms bfs_merge=   2.0ms submit=   2.4ms
cây cân bằng                         ans= 31399782 | memo=   4.3ms pair=   1.7ms iterative=   4.6ms bfs_merge=   2.5ms submit=   1.7ms
cây ngẫu nhiên                       ans= 31266004 | memo=   5.8ms pair=   2.0ms iterative=   5.6ms bfs_merge=   3.1ms submit=   1.9ms
mọi giá trị 0, cây ngẫu nhiên (đáp án 0) ans=        0 | memo=   5.5ms pair=   1.9ms iterative=   4.7ms bfs_merge=   2.4ms submit=   1.5ms
PASS n = 10^4


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ so sánh tổng tầng chẵn với tổng tầng lẻ
Xem `sai_chan_le` ở mục 12.2: với xích `[4, 1, None, 2, None, 3]` (`4 - 1 - 2 - 3`) cho `6` thay vì `7`. **Vì sao sai:** có thể cướp hai nhà cách nhau **hơn một tầng** (`4` và `3`), không nhất thiết cùng tính chẵn lẻ tầng. **Code đúng:** DP hai trạng thái trên từng nút.

### Lỗi 2: Luôn cướp nút hiện tại
#### Code sai

In [17]:
def sai_luon_cuop_goc(root):
    def f(u):
        if u is None:
            return 0
        take = u.val                                     # SAI: không bao giờ cân nhắc bỏ u
        if u.left:
            take += f(u.left.left) + f(u.left.right)
        if u.right:
            take += f(u.right.left) + f(u.right.right)
        return take
    return f(root)


t = build_tree([1, 100])
print("sai:", sai_luon_cuop_goc(t), "| đúng:", sol.rob(t))
assert sai_luon_cuop_goc(t) != sol.rob(t)

sai: 1 | đúng: 100


**Vì sao sai:** đôi khi bỏ nút (để cướp con lớn hơn) tốt hơn. Cây `[1, 100]` nên cướp `100`. **Code đúng:** `max(cướp u, bỏ u)`.

### Lỗi 3: Tính sai trạng thái "bỏ nút"
#### Code sai

In [18]:
def sai_skip(root):
    def dfs(u):
        if u is None:
            return 0, 0
        l_rob, l_skip = dfs(u.left)
        r_rob, r_skip = dfs(u.right)
        rob = u.val + l_skip + r_skip
        skip = l_skip + r_skip                           # SAI: bỏ u thì mỗi con vẫn được chọn cướp (max(rob, skip))
        return rob, skip
    return max(dfs(root))


t = build_tree([3, 4, 5, 1, 3, None, 1])
print("sai:", sai_skip(t), "| đúng:", sol.rob(t))
assert sai_skip(t) != sol.rob(t)

sai: 3 | đúng: 9


**Vì sao sai:** nếu bỏ `u` thì hai con **không còn bị ràng buộc**, chúng có thể cướp (`rob`) hoặc không (`skip`); phải lấy `max(rob, skip)` cho mỗi con. **Code đúng:** `skip = max(l_rob, l_skip) + max(r_rob, r_skip)`.

### Lỗi 4: Ghi nhớ theo giá trị nút thay vì theo nút
#### Code sai

In [19]:
def sai_memo_theo_gia_tri(root):
    memo = {}

    def f(u):
        if u is None:
            return 0
        if u.val in memo:                                # SAI: hai nút khác nhau có thể cùng val
            return memo[u.val]
        take = u.val
        if u.left:
            take += f(u.left.left) + f(u.left.right)
        if u.right:
            take += f(u.right.left) + f(u.right.right)
        memo[u.val] = max(take, f(u.left) + f(u.right))
        return memo[u.val]

    return f(root)


t = build_tree([2, 1, 1, 3])
print("sai:", sai_memo_theo_gia_tri(t), "| đúng:", sol.rob(t), "| oracle:", oracle(t))
assert sai_memo_theo_gia_tri(t) != sol.rob(t)

sai: 6 | đúng: 5 | oracle: 5


**Vì sao sai:** kết quả của một nút phụ thuộc **vị trí/cây con** của nó, không phải giá trị. Hai nút cùng `val = 1` nhưng một nút có con `3` còn nút kia là lá cho kết quả khác nhau. **Code đúng:** khóa là `id(nút)` (hoặc trả cặp từ hàm đệ quy).

### Lỗi 5: Đệ quy không ghi nhớ — bùng nổ số lần gọi
#### Code sai

In [20]:
def rob_naive_count(root):
    calls = 0

    def f(u):
        nonlocal calls
        calls += 1
        if u is None:
            return 0
        take = u.val
        if u.left:
            take += f(u.left.left) + f(u.left.right)
        if u.right:
            take += f(u.right.left) + f(u.right.right)
        return max(take, f(u.left) + f(u.right))

    return f(root), calls


for n in (10, 15, 20, 25):
    t = robber_line_chain([1] * n, "left")
    ans, calls = rob_naive_count(t)
    print(f"xích {n:>2} nút: đáp án {ans:>2} | số lần gọi đệ quy không memo = {calls:>8}")
t = robber_line_chain([1] * 25, "left")
assert rob_naive_count(t)[1] > 100_000

xích 10 nút: đáp án  5 | số lần gọi đệ quy không memo =      463
xích 15 nút: đáp án  8 | số lần gọi đệ quy không memo =     5165
xích 20 nút: đáp án 10 | số lần gọi đệ quy không memo =    57311
xích 25 nút: đáp án 13 | số lần gọi đệ quy không memo =   635619


**Vì sao sai/nguy hiểm:** số lần gọi tăng như Fibonacci khi `n` tăng; với `n = 10^4` không thể chạy. **Code đúng:** hàm trả cặp `(rob, skip)` hoặc memo theo nút.

### Lỗi 6 (hiệu năng): Đệ quy quá sâu
#### Code sai

In [21]:
def goi_voi_gioi_han(fn, root, limit):
    old = sys.getrecursionlimit()
    sys.setrecursionlimit(limit)
    try:
        return fn(root)
    except RecursionError:
        return "RecursionError"
    finally:
        sys.setrecursionlimit(old)


deep = robber_line_chain([1] * 5000, "left")
print("đệ quy, giới hạn 1000 :", goi_voi_gioi_han(sol.rob, deep, 1000))
print("lặp, giới hạn 1000    :", goi_voi_gioi_han(rob_iterative, deep, 1000))
assert goi_voi_gioi_han(sol.rob, deep, 1000) == "RecursionError"
assert goi_voi_gioi_han(rob_iterative, deep, 1000) == 2500

đệ quy, giới hạn 1000 : RecursionError
lặp, giới hạn 1000    : 2500


**Vì sao nguy hiểm:** cây xích sâu `10^4` làm hàm đệ quy vượt giới hạn mặc định. **Code đúng:** nâng `sys.setrecursionlimit` hoặc dùng bản lặp (Cách 4/5).

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**chọn tập nút trên cây, không chọn hai nút kề nhau**, tối đa hóa tổng" → **DP hai trạng thái** (chọn / không chọn) trên mỗi nút (tập độc lập có trọng số lớn nhất trên cây).
- Công thức: `rob = val + skip_L + skip_R`; `skip = max(rob_L, skip_L) + max(rob_R, skip_R)`; đáp án `max(rob, skip)` ở gốc.
- Hàm trả **cặp** giúp tránh tính lặp mà không cần từ điển.
- Cây sâu `10^4`: nâng giới hạn đệ quy hoặc dùng bản lặp/BFS ngược.
- Họ hàng: **House Robber** (198), **House Robber II** (213, vòng tròn), **Binary Tree Cameras** (968), **Maximum Independent Set trên cây**, **Delete and Earn** (740), **Minimum Vertex Cover trên cây**.
- Test: **oracle vét cạn tập con** + **House Robber I** cho xích.
- Thử: `[1,100]`, xích `4-1-2-3`, mọi giá trị bằng nhau, cây sâu `10^4`.

## 15. Biến thể của bài

1. **House Robber** (198) — dãy thẳng (mục 3.1).
2. **House Robber II** (213) — vòng tròn: chạy hai lần (bỏ nhà đầu / bỏ nhà cuối).
3. **Trả về tập nhà cần cướp** (truy vết) thay vì tổng.
4. **Cây `k` nhánh** (mỗi nút nhiều con): `rob = val + Σ skip_child`, `skip = Σ max(rob_child, skip_child)`.
5. **Số nhà tối thiểu để giám sát mọi nhà** (Binary Tree Cameras, 968) — quy hoạch ba trạng thái.
#### Code demo

In [22]:
def rob_circle(a):
    """LeetCode 213: nhà đầu và nhà cuối cũng kề nhau."""
    if len(a) == 1:
        return a[0]
    return max(rob_line(a[1:]), rob_line(a[:-1]))


def rob_tree_with_set(root):
    """Truy vết: trả (tổng, tập các giá trị/nút được cướp)."""
    def dfs(u):
        if u is None:
            return (0, []), (0, [])
        (lr, lrs), (ls, lss) = dfs(u.left)
        (rr, rrs), (rs, rss) = dfs(u.right)
        take = (u.val + ls + rs, [u.val] + lss + rss)
        l_best = (lr, lrs) if lr >= ls else (ls, lss)
        r_best = (rr, rrs) if rr >= rs else (rs, rss)
        skip = (l_best[0] + r_best[0], l_best[1] + r_best[1])
        return take, skip

    take, skip = dfs(root)
    return max(take, skip, key=lambda x: x[0])


def rob_k_ary(children, vals, u=0):
    """Cây nhiều nhánh biểu diễn bằng danh sách con."""
    rob, skip = vals[u], 0
    for v in children[u]:
        r, s = rob_k_ary(children, vals, v)
        rob += s
        skip += max(r, s)
    return rob, skip


print(rob_circle([2, 3, 2]), rob_circle([1, 2, 3, 1]))                 # 3 và 4
print(rob_tree_with_set(build_tree([3, 4, 5, 1, 3, None, 1])))        # (9, [4, 5]) hoặc tương đương
print(max(rob_k_ary({0: [1, 2, 3], 1: [], 2: [], 3: []}, [1, 4, 5, 6])))   # gốc 1 + ... = max(1, 4+5+6) = 15

3 4
(9, [4, 5])
15


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đệ quy không memo | cướp (cháu) / bỏ (con) | mũ | `O(h)` | baseline, oracle `n` rất nhỏ |
| 2. Đệ quy + memo | lưu theo `id(nút)` | `O(n)` | `O(n)` | cầu nối tới DP |
| 3. DP cặp `(rob, skip)` | hậu thứ tự | `O(n)` | `O(h)` | **bản nộp chuẩn** |
| 4. Hậu thứ tự lặp | ngăn xếp tường minh | `O(n)` | `O(n)` | cây rất sâu |
| 5. BFS rồi gộp ngược | đẩy kết quả từ lá lên gốc | `O(n)` | `O(n)` | không đệ quy, đơn giản |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán `sol.rob` cho: (a) `[5]`; (b) `[1, 100]`; (c) `[0, 5, 5]`; (d) `[3, 4, 5]`.

**Bài 2 (tính bằng tay).** Với `[3, 2, 3, None, 3, None, 1]` viết `(rob, skip)` cho từng nút theo hậu thứ tự và kết luận.

**Bài 3 (điền TODO).** Hoàn thiện `rob_todo(root)` theo Cách 3.
#### Code demo

In [23]:
def rob_todo(root):
    # TODO 1: viết dfs(u) trả (rob, skip); cây rỗng -> (0, 0)
    # TODO 2: rob = u.val + l_skip + r_skip
    # TODO 3: skip = max(l_rob, l_skip) + max(r_rob, r_skip)
    # TODO 4: return max(dfs(root))
    pass


# Tự làm xong rồi mới mở test:
# for values, expected in examples: assert rob_todo(build_tree(values)) == expected

**Bài 4 (tự viết lại).** Viết `rob_with_forbidden(root, forbidden)`: giống bài gốc nhưng một số nhà (theo **giá trị nút** nằm trong `forbidden`, giả sử mọi giá trị khác nhau) **không được cướp**. Gợi ý: nút bị cấm ép `rob = -∞`.

**Bài 5 (biến thể / mini challenge).** Viết `rob_at_most_k(root, k)`: cướp **tối đa `k` nhà** (không báo động), tối đa hóa tổng. Gợi ý: DP với thêm chiều "số nhà đã chọn" (`dp[u][j][0/1]`), gộp hai con bằng tích chập `O(k²)`. Thử trên cây nhỏ và so với vét cạn.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: không cướp cha–con cùng lúc; Bài 2: lá có `(val, 0)`. Bài 4: nút cấm không thể cướp. Bài 5: thêm chiều đếm số nhà.
- **Mức 2 (vừa):** Bài 1: `5`, `100`, `10`, `9` (`4 + 5`). Bài 2: các lá `(3,0)`, `(1,0)`; nút `2 = (2, 3)`; nút `3` phải `= (3, 1)`; gốc `= (7, 6)` ⇒ `7`. Bài 3: khung Cách 3.
- **Mức 3 (sâu):** Bài 4: dùng `NEG = float('-inf')`; nút cấm: `rob = NEG`, vẫn tính `skip` bình thường; đáp án `max(rob, skip)` hữu hạn vì `skip` luôn hữu hạn. Bài 5: `dp[u][j][1]` (cướp `u`, đã chọn `j` nhà trong cây con) và `dp[u][j][0]`; gộp con bằng `new[a + b] = max(left[a] + right[b])`; chú ý `rob_at_most_k` là `max` theo `j ≤ k`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [24]:
# Bài 1
assert [sol.rob(build_tree(v)) for v in ([5], [1, 100], [0, 5, 5], [3, 4, 5])] == [5, 100, 10, 9]

# Bài 2
assert sol.rob(build_tree([3, 2, 3, None, 3, None, 1])) == 7


# Bài 3
def rob_todo_ans(root):
    def dfs(u):
        if u is None:
            return 0, 0
        l_rob, l_skip = dfs(u.left)
        r_rob, r_skip = dfs(u.right)
        return u.val + l_skip + r_skip, max(l_rob, l_skip) + max(r_rob, r_skip)

    return max(dfs(root))


for values, expected in examples:
    assert rob_todo_ans(build_tree(values)) == expected


# Bài 4
def rob_with_forbidden(root, forbidden):
    NEG = float("-inf")

    def dfs(u):
        if u is None:
            return 0, 0
        l_rob, l_skip = dfs(u.left)
        r_rob, r_skip = dfs(u.right)
        rob = NEG if u.val in forbidden else u.val + l_skip + r_skip
        skip = max(l_rob, l_skip) + max(r_rob, r_skip)
        return rob, skip

    return max(dfs(root))


def oracle_forbidden(root, forbidden):
    order, parent = all_nodes_with_parent(root)
    n = len(order)
    best = 0
    for mask in range(1 << n):
        ok = True
        for i in range(n):
            if (mask >> i) & 1 and order[i].val in forbidden:
                ok = False
                break
        if not ok or any((mask >> i) & 1 and (mask >> parent[i]) & 1 for i in range(1, n)):
            continue
        best = max(best, sum(order[i].val for i in range(n) if (mask >> i) & 1))
    return best


rng = random.Random(77)
for _ in range(200):
    n = rng.randint(1, 9)
    vals = rng.sample(range(1, 30), n)                    # giá trị khác nhau
    t = build_tree([vals[0]])
    nodes = [t]
    slots = [(t, "left"), (t, "right")]
    for v in vals[1:]:
        idx = rng.randrange(len(slots))
        slots[idx], slots[-1] = slots[-1], slots[idx]
        parent, side = slots.pop()
        child = TreeNode(v)
        setattr(parent, side, child)
        slots.extend([(child, "left"), (child, "right")])
    forb = set(rng.sample(vals, rng.randint(0, n)))
    assert rob_with_forbidden(t, forb) == oracle_forbidden(t, forb)


# Bài 5
def rob_at_most_k(root, k):
    NEG = float("-inf")

    def merge(a, b):
        res = [NEG] * (k + 1)
        for i, x in enumerate(a):
            if x == NEG:
                continue
            for j in range(k + 1 - i):
                if b[j] != NEG:
                    res[i + j] = max(res[i + j], x + b[j])
        return res

    def dfs(u):
        # w[j]: tốt nhất khi KHÔNG cướp u và chọn đúng j nhà trong cây con; t[j]: khi cướp u
        if u is None:
            return [0] + [NEG] * k, [NEG] * (k + 1)
        lw, lt = dfs(u.left)
        rw, rt = dfs(u.right)
        l_any = [max(a, b) for a, b in zip(lw, lt)]
        r_any = [max(a, b) for a, b in zip(rw, rt)]
        w = merge(l_any, r_any)
        inner = merge(lw, rw)                                # cướp u thì hai con đều không được cướp
        t = [NEG] * (k + 1)
        for j in range(1, k + 1):
            if inner[j - 1] != NEG:
                t[j] = inner[j - 1] + u.val
        return w, t

    w, t = dfs(root)
    return max(max(w), max(t))


def oracle_at_most_k(root, k):
    order, parent = all_nodes_with_parent(root)
    n = len(order)
    best = 0
    for mask in range(1 << n):
        if bin(mask).count("1") > k:
            continue
        if any((mask >> i) & 1 and (mask >> parent[i]) & 1 for i in range(1, n)):
            continue
        best = max(best, sum(order[i].val for i in range(n) if (mask >> i) & 1))
    return best


rng = random.Random(78)
for _ in range(200):
    t = random_tree(rng, rng.randint(1, 9), 9)
    k = rng.randint(0, 5)
    assert rob_at_most_k(t, k) == oracle_at_most_k(t, k), (to_level_order(t), k)
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (DP cặp `(rob, skip)`)

```python
class Solution:
    def rob(self, root: Optional[TreeNode]) -> int:
        def dfs(node):
            if node is None:
                return 0, 0
            l_rob, l_skip = dfs(node.left)
            r_rob, r_skip = dfs(node.right)
            rob = node.val + l_skip + r_skip
            skip = max(l_rob, l_skip) + max(r_rob, r_skip)
            return rob, skip

        return max(dfs(root))
```

**Checklist**

- [ ] Mỗi nút có **hai trạng thái**: cướp / bỏ?
- [ ] Cướp `node` ⇒ cộng `skip` của hai con (không phải `max`)?
- [ ] Bỏ `node` ⇒ mỗi con lấy `max(rob, skip)`?
- [ ] Đáp án `max(rob, skip)` ở gốc?
- [ ] Cây rỗng `(0, 0)`?
- [ ] Không dùng memo theo giá trị; không chỉ tách tầng chẵn/lẻ?
- [ ] Cây sâu `10^4`: nâng giới hạn đệ quy hoặc dùng bản lặp?
- [ ] Skeleton đúng `class Solution` / `def rob(self, root: Optional[TreeNode]) -> int`?